The code below creates an interactive visualization from the selected network data and solution data. 

Requirements:
- Network data. This is the json data file acquired from the instance generator.
- Solution data. This is the json data file acquired after solving the problem with one of the solvers. Preferably CPPMTO_optimizer1.
- Clustered network data frame. This is the pickle file that has the dataframe information for the network. The dataframe for this is created after running map_and_hub_creator.py, which creates the clusters from the network data. Note that the same data frame is an argument of the instance generator. 

In [ ]:
import os
import ipywidgets as widgets
from IPython.display import display
from IPython.display import clear_output
import pickle
import geopandas as gpd
from Vis_Codes.visualize_bilevel_solution_with_ui import display_bilevel_solution_with_ui
from summarizer import summarize_experiment_solutions

# Define the folder to search. For simplicity, use the current directory.
folder = os.getcwd()  # You can also specify a custom path here.

# Get all JSON files in the folder.
json_files = [f for f in os.listdir(folder) if f.endswith('.json')]

# Split JSON files into two groups:
#   - preproc_files: those that do NOT contain "solution" (case-insensitive)
#   - solution_files: those that DO contain "solution"
preproc_files = [f for f in json_files if "solution" not in f.lower()]
solution_files  = [f for f in json_files if "solution" in f.lower()]

# Get all pickle files.
pickle_files = [f for f in os.listdir(folder) if f.endswith('.pkl')]

# Create dropdown widgets for each file type.
preproc_dropdown = widgets.Dropdown(options=preproc_files, description='Preproc JSON:')
solution_dropdown = widgets.Dropdown(options=solution_files, description='Solution JSON:')
pickle_dropdown = widgets.Dropdown(options=pickle_files, description='Hub Pickle:')

# Add widgets for origin and destination input.
origin_widget = widgets.IntText(value=4, description='Origin:')
destination_widget = widgets.IntText(value=5, description='Destination:')

# Create a button widget to trigger visualization.
visualize_button = widgets.Button(description="Visualize")

map_output = widgets.Output()

def on_visualize_clicked(b):
    with map_output:
        map_output.clear_output(wait = True)  # Clear previous output
        # Build full paths.
        preproc_path = os.path.join(folder, preproc_dropdown.value)
        solution_path = os.path.join(folder, solution_dropdown.value)
        pickle_path = os.path.join(folder, pickle_dropdown.value)
        
        # Load hub_df from the selected pickle file.
        with open(pickle_path, "rb") as f:
            hub_df = pickle.load(f)
        # Ensure hub_df is a GeoDataFrame with proper CRS.
        hub_df = gpd.GeoDataFrame(hub_df, geometry='geometry', crs="EPSG:3857")

        # Call your visualization function.
        # (Assuming display_bilevel_solution_with_ui is imported from your module.)
        m = display_bilevel_solution_with_ui(hub_df, solution_path, preproc_path)
        
        # Save the map as an HTML file.
        #m.save("output_map.html")
        #print("Map saved as output_map.html")
        
        # Display the map.
        display(m)

visualize_button.on_click(on_visualize_clicked)

# Arrange the widgets in a vertical box.
ui = widgets.VBox([preproc_dropdown, solution_dropdown, pickle_dropdown, visualize_button])
display(ui, map_output)

disp_df=summarize_experiment_solutions(folder)


Output()

KeyError: 'Total PT Only Arcs'